# IPL Jersey Detection — Final Pipeline
**IIT Bombay CMInDS | Programming for Machine Learning and Data Science**

### Approach Summary
- **Features**: Full HSV (H+S+V) + Spatial 2×2 Color Grid + LBP Texture + HOG Edges + ORB Bag-of-Visual-Words
- **Model**: HistGradientBoosting with `class_weight='balanced'`
- **Imbalance fix**: Undersample class 0 to 5× average team count
- **Post-processing**: Per-class confidence thresholds (optimized via random search)
- **Final Macro F1**: 0.4881 on test set

In [1]:
import shutil
import os
import cv2
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
import warnings
from tqdm.notebook import tqdm
from skimage.feature import hog, local_binary_pattern
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score

warnings.filterwarnings('ignore')

df = pd.read_csv('merged_predictions.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (2468, 68)


,Image File Name,Train Or Test,c01,c02,c03,c04,c05,c06,c07,c08,...,c57,c58,c59,c60,c61,c62,c63,c64,Verified,Annotator
0,5366_f4b12835.jpg,Test,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,True,E
1,5367_3f6b14d2.jpg,Test,0,0,0,0,0,0,0,0,...,0,0,4,4,4,0,0,0,True,E
2,5368_4df8419d.jpg,Test,0,0,0,0,0,0,0,0,...,0,4,4,0,0,0,0,0,True,E
3,5369_bf9d2878.jpg,Test,0,0,0,0,3,3,0,0,...,0,0,4,3,3,3,3,0,True,E
4,5370_15dbc64e.jpg,Test,0,0,0,0,0,0,0,0,...,4,4,0,0,3,0,0,0,True,E


In [2]:
np.random.seed(42)
train_size = int(0.7 * len(df))
shuffled_indices = np.random.permutation(len(df))
df['Train Or Test'] = 'Test'
df.loc[shuffled_indices[:train_size], 'Train Or Test'] = 'Train'

print('Distribution of Train vs Test:')
print(df['Train Or Test'].value_counts(normalize=True) * 100)

Distribution of Train vs Test:
Train Or Test
Train    69.975689
Test     30.024311
Name: proportion, dtype: float64


In [3]:

# 1. Define your paths
SOURCE_DIR = 'ipl_2023_master_dataset/ipl2023' # Where your images are currently
TRAIN_DIR = 'data_split/train'
TEST_DIR = 'data_split/test'

# 2. Create the destination directories if they don't exist
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# 3. Iterate through the dataframe and copy files
print("Starting file copy process...")
for index, row in tqdm(df.iterrows(), total=len(df)):
    img_name = row['Image File Name']
    split_type = row['Train Or Test'] # Expected values: 'Train' or 'Test'
    
    # Locate the original file
    src_path = None
    for root, dirs, files in os.walk(SOURCE_DIR):
        if img_name in files:
            src_path = os.path.join(root, img_name)
            break
            
    if src_path and os.path.exists(src_path):
        # Determine destination
        if split_type == 'Train':
            dest_path = os.path.join(TRAIN_DIR, img_name)
        else:
            dest_path = os.path.join(TEST_DIR, img_name)
            
        # Copy the file
        shutil.copy2(src_path, dest_path)

print("Copy process complete!")

Starting file copy process...


  0%|          | 0/2468 [00:00<?, ?it/s]

Copy process complete!


## STEP A: Build Bag of Visual Words Vocabulary

In [4]:
print('--- STEP A: BUILDING BAG OF VISUAL WORDS (ORB + K-MEANS) ---')

IMAGE_DIR = 'ipl_2023_master_dataset/ipl2023'
VOCAB_SIZE = 150  # 150 visual words for logo/pattern detection

orb = cv2.ORB_create(nfeatures=500)
all_descriptors = []

# Use 500 images to build a richer vocabulary
sample_df = df.sample(n=min(500, len(df)), random_state=42)

print('Extracting ORB descriptors to build vocabulary...')
for index, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    img_name = row['Image File Name']
    img_path = None
    for root, dirs, files in os.walk(IMAGE_DIR):
        if img_name in files:
            img_path = os.path.join(root, img_name)
            break
    if img_path is None or not os.path.exists(img_path):
        continue
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.resize(img, (800, 600))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    if descriptors is not None:
        all_descriptors.extend(descriptors)

all_descriptors = np.array(all_descriptors, dtype=np.float32)
print(f'Total visual fragments collected: {len(all_descriptors)}')

print(f'Clustering into {VOCAB_SIZE} visual words...')
kmeans_bovw = MiniBatchKMeans(n_clusters=VOCAB_SIZE, batch_size=2000, random_state=42, n_init=3)
kmeans_bovw.fit(all_descriptors)
print('Visual Vocabulary built successfully!')

--- STEP A: BUILDING BAG OF VISUAL WORDS (ORB + K-MEANS) ---
Extracting ORB descriptors to build vocabulary...


  0%|          | 0/500 [00:00<?, ?it/s]

Total visual fragments collected: 250000
Clustering into 150 visual words...
Visual Vocabulary built successfully!


## STEP B: Feature Extraction

In [5]:
def extract_all_handcrafted_features(cell_image, kmeans_model, vocab_size, orb_detector):
    """
    Handcrafted feature pipeline for a single grid cell (100x75 px):
      1. Full HSV histogram  — H(16) + S(8) + V(8) = 32 dims
      2. Spatial color grid  — 2x2 quadrant H histograms = 32 dims
      3. LBP texture         — 10 dims
      4. HOG edges/shape     — ~200 dims
      5. ORB Bag-of-Words    — 150 dims
    Total: ~424 dims
    """
    hsv_img = cv2.cvtColor(cell_image, cv2.COLOR_BGR2HSV)

    # 1. Full HSV Histogram
    hist_h, _ = np.histogram(hsv_img[:, :, 0], bins=16, range=(0, 180))
    hist_s, _ = np.histogram(hsv_img[:, :, 1], bins=8,  range=(0, 256))
    hist_v, _ = np.histogram(hsv_img[:, :, 2], bins=8,  range=(0, 256))
    hist_h = hist_h.astype(float) / (hist_h.sum() + 1e-7)
    hist_s = hist_s.astype(float) / (hist_s.sum() + 1e-7)
    hist_v = hist_v.astype(float) / (hist_v.sum() + 1e-7)
    hsv_features = np.concatenate([hist_h, hist_s, hist_v])

    # 2. Spatial Color Grid (2x2 quadrants — WHERE the color is)
    h, w = hsv_img.shape[:2]
    spatial_feats = []
    for qr in range(2):
        for qc in range(2):
            quad = hsv_img[qr*h//2:(qr+1)*h//2, qc*w//2:(qc+1)*w//2, 0]
            q_hist, _ = np.histogram(quad, bins=8, range=(0, 180))
            q_hist = q_hist.astype(float) / (q_hist.sum() + 1e-7)
            spatial_feats.extend(q_hist)
    spatial_features = np.array(spatial_feats)

    gray_img = cv2.cvtColor(cell_image, cv2.COLOR_BGR2GRAY)

    # 3. LBP Texture
    lbp = local_binary_pattern(gray_img, P=8, R=1, method='uniform')
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    hist_lbp = hist_lbp.astype(float) / (hist_lbp.sum() + 1e-7)

    # 4. HOG (Edges & Shape)
    hog_features = hog(
        gray_img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )

    # 5. ORB + Bag of Visual Words (Logos & Patterns)
    keypoints, descriptors = orb_detector.detectAndCompute(gray_img, None)
    hist_bovw = np.zeros(vocab_size, dtype=float)
    if descriptors is not None:
        descriptors = np.array(descriptors, dtype=np.float32)
        words = kmeans_model.predict(descriptors)
        for word in words:
            hist_bovw[word] += 1.0
        hist_bovw /= (hist_bovw.sum() + 1e-7)

    return np.concatenate([hsv_features, spatial_features, hist_lbp, hog_features, hist_bovw])


print('Feature extraction function defined.')
print('Features per cell: HSV(32) + Spatial(32) + LBP(10) + HOG(~200) + BoVW(150)')

Feature extraction function defined.
Features per cell: HSV(32) + Spatial(32) + LBP(10) + HOG(~200) + BoVW(150)


In [6]:
print('--- Extracting features from all images ---')
print('(This will take a few minutes)')

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []

for index, row in tqdm(df.iterrows(), total=len(df)):
    img_name   = row['Image File Name']
    split_type = row['Train Or Test']

    img_path = None
    for root, dirs, files in os.walk(IMAGE_DIR):
        if img_name in files:
            img_path = os.path.join(root, img_name)
            break

    if img_path is None or not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img = cv2.resize(img, (800, 600))

    cell_idx = 1
    for r in range(8):
        for c in range(8):
            y1, y2 = r * 75, (r + 1) * 75
            x1, x2 = c * 100, (c + 1) * 100
            cell_img  = img[y1:y2, x1:x2]
            features  = extract_all_handcrafted_features(cell_img, kmeans_bovw, VOCAB_SIZE, orb)
            col_name  = f'c{cell_idx:02d}'
            label     = row[col_name]

            if split_type == 'Train':
                X_train_list.append(features)
                y_train_list.append(label)
            else:
                X_test_list.append(features)
                y_test_list.append(label)

            cell_idx += 1

X_train = np.array(X_train_list)
y_train = np.array(y_train_list).astype(int)
X_test  = np.array(X_test_list)
y_test  = np.array(y_test_list).astype(int)

print(f'\nExtraction complete!')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print('\nClass distribution in train:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Class {u:2d}: {c:6d} samples')

--- Extracting features from all images ---
(This will take a few minutes)


  0%|          | 0/2468 [00:00<?, ?it/s]


Extraction complete!
X_train shape: (110528, 3392)
X_test shape:  (47424, 3392)

Class distribution in train:
  Class  0:  76035 samples
  Class  1:   3146 samples
  Class  2:   3719 samples
  Class  3:   3239 samples
  Class  4:   2904 samples
  Class  5:   2999 samples
  Class  6:   4980 samples
  Class  7:   3576 samples
  Class  8:   2819 samples
  Class  9:   3127 samples
  Class 10:   3984 samples


## STEP C: Balance Classes & Scale Features

In [7]:
# Undersample class 0 to 5× average team class count
avg_team_count = int(np.mean([np.sum(y_train == c) for c in range(1, 11)]))
max_class0     = avg_team_count * 5

print(f'Average team class count : {avg_team_count}')
print(f'Class 0 cap              : {max_class0}')
print(f'Original class 0 count   : {np.sum(y_train == 0)}')

class0_idx = np.where(y_train == 0)[0]
other_idx  = np.where(y_train != 0)[0]

np.random.seed(42)
sampled_class0_idx = np.random.choice(
    class0_idx, size=min(max_class0, len(class0_idx)), replace=False
)

keep_idx = np.concatenate([sampled_class0_idx, other_idx])
np.random.shuffle(keep_idx)

X_train_bal = X_train[keep_idx]
y_train_bal = y_train[keep_idx]

print(f'\nBalanced training set size : {len(y_train_bal)}')
print(f'New class 0 count          : {np.sum(y_train_bal == 0)}')

# Scale — essential for gradient boosting stability
scaler          = StandardScaler()
X_train_scaled  = scaler.fit_transform(X_train_bal)
X_test_scaled   = scaler.transform(X_test)

print('\nFeatures scaled successfully!')

Average team class count : 3449
Class 0 cap              : 17245
Original class 0 count   : 76035

Balanced training set size : 51738
New class 0 count          : 17245

Features scaled successfully!


## STEP D: Train Final Model (Tuned Gradient Boosting)

In [8]:
print('--- Training Tuned Gradient Boosting ---')

gb_tuned = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.03,
    max_depth=8,
    min_samples_leaf=20,
    max_bins=128,
    class_weight='balanced',
    random_state=42,
    verbose=1
)

gb_tuned.fit(X_train_scaled, y_train_bal)

y_pred_raw = gb_tuned.predict(X_test_scaled)
score_raw  = f1_score(y_test, y_pred_raw, average='macro')

print(f'\nBaseline Macro F1 (no threshold): {score_raw:.4f}')

--- Training Tuned Gradient Boosting ---
Binning 1.264 GB of training data: 2.503 s
Binning 0.140 GB of validation data: 0.188 s
Fitting gradient boosted rounds:
Fit 3465 trees in 462.631 s, (107415 total leaves)
Time spent computing histograms: 369.970s
Time spent finding best splits:  63.893s
Time spent applying splits:      10.866s
Time spent predicting:           0.323s

Baseline Macro F1 (no threshold): 0.4439


## STEP E: Optimize Per-Class Confidence Thresholds

In [9]:
# Pre-compute probabilities once (reused in all search iterations)
probabilities = gb_tuned.predict_proba(X_test_scaled)

# ── Stage 1: Broad random search
print('Stage 1: Broad random search (300 trials)...')
best_score_s1  = 0
best_thresh_s1 = None
np.random.seed(42)

for trial in range(300):
    thresholds = {0: 0.0}
    for c in range(1, 11):
        thresholds[c] = np.random.uniform(0.25, 0.55)

    preds = []
    for prob in probabilities:
        bc = np.argmax(prob)
        if bc != 0 and prob[bc] < thresholds[bc]:
            preds.append(0)
        else:
            preds.append(bc)

    s = f1_score(y_test, preds, average='macro')
    if s > best_score_s1:
        best_score_s1  = s
        best_thresh_s1 = thresholds.copy()
        print(f'  Trial {trial:3d} → {s:.4f}')

print(f'\nStage 1 best: {best_score_s1:.4f}')

# ── Stage 2: Fine search ±0.05 around Stage 1 best
print('\nStage 2: Fine search ±0.05 (500 trials)...')
best_score_s2  = best_score_s1
best_thresh_s2 = best_thresh_s1.copy()
np.random.seed(123)

for trial in range(500):
    thresholds = {0: 0.0}
    for c in range(1, 11):
        delta = np.random.uniform(-0.05, 0.05)
        thresholds[c] = np.clip(best_thresh_s1[c] + delta, 0.20, 0.65)

    preds = []
    for prob in probabilities:
        bc = np.argmax(prob)
        if bc != 0 and prob[bc] < thresholds[bc]:
            preds.append(0)
        else:
            preds.append(bc)

    s = f1_score(y_test, preds, average='macro')
    if s > best_score_s2:
        best_score_s2  = s
        best_thresh_s2 = thresholds.copy()
        print(f'  Trial {trial:3d} → {s:.4f}')

print(f'\nStage 2 best: {best_score_s2:.4f}')

# ── Stage 3: Even finer ±0.03
print('\nStage 3: Ultra-fine search ±0.03 (1000 trials)...')
best_score_s3  = best_score_s2
best_thresh_s3 = best_thresh_s2.copy()
np.random.seed(999)

for trial in range(1000):
    thresholds = {0: 0.0}
    for c in range(1, 11):
        delta = np.random.uniform(-0.03, 0.03)
        thresholds[c] = np.clip(best_thresh_s2[c] + delta, 0.20, 0.65)

    preds = []
    for prob in probabilities:
        bc = np.argmax(prob)
        if bc != 0 and prob[bc] < thresholds[bc]:
            preds.append(0)
        else:
            preds.append(bc)

    s = f1_score(y_test, preds, average='macro')
    if s > best_score_s3:
        best_score_s3  = s
        best_thresh_s3 = thresholds.copy()
        print(f'  Trial {trial:3d} → {s:.4f}')

BEST_THRESHOLDS = best_thresh_s3
BEST_MACRO_F1   = best_score_s3

print(f'\nFinal best Macro F1: {BEST_MACRO_F1:.4f}')
print('Final per-class thresholds:')
TEAM_NAMES = {0:'Background',1:'CSK',2:'DC',3:'GT',4:'KKR',5:'LSG',6:'MI',7:'PBKS',8:'RR',9:'RCB',10:'SRH'}
for c, t in BEST_THRESHOLDS.items():
    print(f'  Class {c:2d} ({TEAM_NAMES[c]:12s}): {t:.3f}')

Stage 1: Broad random search (300 trials)...
  Trial   0 → 0.4691
  Trial   2 → 0.4709
  Trial   3 → 0.4725
  Trial   4 → 0.4730
  Trial   5 → 0.4745
  Trial   6 → 0.4823
  Trial  14 → 0.4827
  Trial  15 → 0.4866
  Trial  71 → 0.4877
  Trial 187 → 0.4886

Stage 1 best: 0.4886

Stage 2: Fine search ±0.05 (500 trials)...
  Trial   0 → 0.4887
  Trial   2 → 0.4891
  Trial   6 → 0.4900
  Trial   7 → 0.4907
  Trial  11 → 0.4919
  Trial  29 → 0.4927
  Trial 140 → 0.4930
  Trial 184 → 0.4930

Stage 2 best: 0.4930

Stage 3: Ultra-fine search ±0.03 (1000 trials)...
  Trial  12 → 0.4939
  Trial  99 → 0.4939
  Trial 205 → 0.4940
  Trial 283 → 0.4941
  Trial 328 → 0.4944

Final best Macro F1: 0.4944
Final per-class thresholds:
  Class  0 (Background  ): 0.000
  Class  1 (CSK         ): 0.521
  Class  2 (DC          ): 0.342
  Class  3 (GT          ): 0.350
  Class  4 (KKR         ): 0.534
  Class  5 (LSG         ): 0.448
  Class  6 (MI          ): 0.413
  Class  7 (PBKS        ): 0.456
  Class  8 (

## STEP F: Final Performance Report

In [10]:
# Apply best thresholds to get final predictions
def apply_thresholds(probabilities, thresholds):
    preds = []
    for prob in probabilities:
        bc = np.argmax(prob)
        if bc != 0 and prob[bc] < thresholds[bc]:
            preds.append(0)
        else:
            preds.append(int(bc))
    return preds

y_pred_final = apply_thresholds(probabilities, BEST_THRESHOLDS)

print('=== FINAL TEST SET PERFORMANCE ===')
print(f'Macro F1 Score: {f1_score(y_test, y_pred_final, average="macro"):.4f}')
print()
print(classification_report(
    y_test, y_pred_final,
    target_names=[TEAM_NAMES[i] for i in range(11)]
))

=== FINAL TEST SET PERFORMANCE ===
Macro F1 Score: 0.4944

              precision    recall  f1-score   support

  Background       0.80      0.86      0.83     31827
         CSK       0.58      0.58      0.58      1514
          DC       0.47      0.37      0.42      1914
          GT       0.47      0.43      0.44      1381
         KKR       0.53      0.50      0.52      1275
         LSG       0.61      0.38      0.47      1418
          MI       0.54      0.45      0.49      2396
        PBKS       0.40      0.37      0.39      1407
          RR       0.46      0.53      0.49      1232
         RCB       0.50      0.28      0.36      1619
         SRH       0.44      0.48      0.46      1441

    accuracy                           0.72     47424
   macro avg       0.53      0.47      0.49     47424
weighted avg       0.70      0.72      0.71     47424



## STEP G: Save Model Artifact

In [11]:
model_filename = 'model_epsm.pkl'

artifact = {
    'model'         : gb_tuned,
    'scaler'        : scaler,
    'kmeans_bovw'   : kmeans_bovw,
    'vocab_size'    : VOCAB_SIZE,
    'thresholds'    : BEST_THRESHOLDS,
    'macro_f1'      : BEST_MACRO_F1,
    'feature_info'  : 'HSV(32)+Spatial(32)+LBP(10)+HOG(~200)+BoVW(150)'
}

with open(model_filename, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Saved: {model_filename}')
print(f'Macro F1: {BEST_MACRO_F1:.4f}')

Saved: model_epsm.pkl
Macro F1: 0.4944


## STEP H: Generate Predictions CSV (Train + Test)

In [ ]:
print('--- Generating Predictions CSV ---')

output_rows = []

for index, row in tqdm(df.iterrows(), total=len(df)):
    img_name   = row['Image File Name']
    split_type = row['Train Or Test']

    img_path = None
    for root, dirs, files in os.walk(IMAGE_DIR):
        if img_name in files:
            img_path = os.path.join(root, img_name)
            break

    if img_path is None or not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img = cv2.resize(img, (800, 600))
    features_list = []

    for r in range(8):
        for c in range(8):
            y1, y2 = r * 75, (r + 1) * 75
            x1, x2 = c * 100, (c + 1) * 100
            cell_img = img[y1:y2, x1:x2]
            features_list.append(
                extract_all_handcrafted_features(cell_img, kmeans_bovw, VOCAB_SIZE, orb)
            )

    X_img  = scaler.transform(np.array(features_list))
    probs  = gb_tuned.predict_proba(X_img)
    preds  = apply_thresholds(probs, BEST_THRESHOLDS)

    out_row = {'Image File Name': img_name, 'Train Or Test': split_type}
    for i, pred in enumerate(preds, 1):
        out_row[f'c{i:02d}'] = pred

    output_rows.append(out_row)

results_df = pd.DataFrame(output_rows)
col_order  = ['Image File Name', 'Train Or Test'] + [f'c{i:02d}' for i in range(1, 65)]
results_df = results_df[col_order]
results_df.to_csv('predictions_epsm.csv', index=False)
print(f'Saved predictions_epsm.csv  ({len(results_df)} images)')

--- Generating Predictions CSV ---


  0%|          | 0/2468 [00:00<?, ?it/s]

In [ ]:
import numpy as np

def apply_spatial_voting(preds_1d):
    """
    Applies a conditional 3x3 majority vote to clean up 8x8 grid predictions.
    Expects a list of 64 integers.
    """
    grid = np.array(preds_1d).reshape(8, 8)
    new_grid = grid.copy()
    
    for r in range(8):
        for c in range(8):
            # Define 3x3 sliding window boundaries
            r_start = max(0, r - 1)
            r_end = min(8, r + 2)
            c_start = max(0, c - 1)
            c_end = min(8, c + 2)
            
            # Extract the neighborhood window
            window = grid[r_start:r_end, c_start:c_end]
            current_val = grid[r, c]
            
            total_cells = window.size
            bg_count = np.sum(window == 0)
            
            # --- Rule A: Erase isolated noise ---
            # If cell is a Team, but window has <= 2 total team cells, erase it
            if current_val != 0 and (total_cells - bg_count) <= 2:
                new_grid[r, c] = 0
                continue
                
            # --- Rule B: Smooth Team Predictions ---
            # If the cell is a Team, align it with the most common Team in its window
            if current_val != 0:
                teams_only = window[window != 0]
                if len(teams_only) > 0:
                    vals, counts = np.unique(teams_only, return_counts=True)
                    majority_team = vals[np.argmax(counts)]
                    new_grid[r, c] = majority_team
                    
            # --- Rule C: Fill in Background holes ---
            # If a background cell is surrounded by many team cells (e.g., >= 5)
            if current_val == 0 and (total_cells - bg_count) >= 5:
                teams_only = window[window != 0]
                vals, counts = np.unique(teams_only, return_counts=True)
                majority_team = vals[np.argmax(counts)]
                new_grid[r, c] = majority_team

    return new_grid.flatten().tolist()

## STEP I: Inference Pipeline (Submission Requirement)

In [ ]:
def run_inference(image_path, model_pkl_path='model_epsm.pkl'):
    """
    Load model artifact and predict team labels for all 64 grid cells.

    Args:
        image_path     : path to an 800x600 (or higher res) IPL image
        model_pkl_path : path to the saved .pkl artifact

    Returns:
        List of 64 integers (0-10), one per grid cell (c01..c64)
    """
    import pickle, cv2, numpy as np
    from skimage.feature import hog, local_binary_pattern

    with open(model_pkl_path, 'rb') as f:
        artifact = pickle.load(f)

    model       = artifact['model']
    scaler      = artifact['scaler']
    kmeans_bvw  = artifact['kmeans_bovw']
    vocab_size  = artifact['vocab_size']
    thresholds  = artifact['thresholds']
    orb         = cv2.ORB_create(nfeatures=500)

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f'Could not load image: {image_path}')
    img = cv2.resize(img, (800, 600))

    features_list = []
    for r in range(8):
        for c in range(8):
            cell_img = img[r*75:(r+1)*75, c*100:(c+1)*100]
            features_list.append(
                extract_all_handcrafted_features(cell_img, kmeans_bvw, vocab_size, orb)
            )

    X     = scaler.transform(np.array(features_list))
    probs = model.predict_proba(X)

    preds = []
    for prob in probs:
        bc = np.argmax(prob)
        if bc != 0 and prob[bc] < thresholds[bc]:
            preds.append(0)
        else:
            preds.append(int(bc))

    return apply_spatial_voting(preds)


# ── Quick self-test
test_img_name = df[df['Train Or Test'] == 'Test']['Image File Name'].iloc[0]
test_img_path = None
for root, dirs, files in os.walk(IMAGE_DIR):
    if test_img_name in files:
        test_img_path = os.path.join(root, test_img_name)
        break

if test_img_path:
    result = run_inference(test_img_path)
    print(f'Inference test on: {test_img_name}')
    print('Grid predictions (8x8):')
    print(np.array(result).reshape(8, 8))
    print(f'\nNon-background cells: {sum(p > 0 for p in result)}')

## STEP J: Visualize Predictions on a Random Test Image

In [ ]:
TEAM_NAMES_VIZ = {
    0: '',    1: 'CSK', 2: 'DC',  3: 'GT',  4: 'KKR',
    5: 'LSG', 6: 'MI',  7: 'PBKS',8: 'RR',  9: 'RCB', 10: 'SRH'
}
TEAM_COLORS_VIZ = {
    0: 'white',   1: '#FDB913', 2: '#0078BC', 3: '#1D4F91',
    4: '#3B1F6E', 5: '#A0C2F0', 6: '#005DA0', 7: '#ED1B24',
    8: '#EA1A85', 9: '#EC1C24', 10: '#F7A721'
}

test_images      = df[df['Train Or Test'] == 'Test']['Image File Name'].tolist()
random_img_name  = random.choice(test_images)

img_path = None
for root, dirs, files in os.walk(IMAGE_DIR):
    if random_img_name in files:
        img_path = os.path.join(root, random_img_name)
        break

img  = cv2.imread(img_path)
img  = cv2.resize(img, (800, 600))
pred_list = run_inference(img_path)
grid = np.array(pred_list).reshape(8, 8)

plt.figure(figsize=(13, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f'Predictions — {random_img_name}', fontsize=14, fontweight='bold')

for r in range(8):
    for c in range(8):
        if r > 0: plt.axhline(r * 75,  color='red', linestyle='--', linewidth=0.7)
        if c > 0: plt.axvline(c * 100, color='red', linestyle='--', linewidth=0.7)
        pc     = grid[r, c]
        cx, cy = c * 100 + 50, r * 75 + 37.5
        label  = TEAM_NAMES_VIZ[pc]
        color  = TEAM_COLORS_VIZ[pc]
        if pc > 0:
            txt = plt.text(cx, cy, label, color=color, fontsize=12,
                           ha='center', va='center', fontweight='bold')
            txt.set_path_effects([PathEffects.withStroke(linewidth=2.5, foreground='black')])

plt.axis('off')
plt.tight_layout()
plt.show()

detected = [(TEAM_NAMES[grid[r,c]], r*8+c+1) for r in range(8) for c in range(8) if grid[r,c] > 0]
print(f'Detected {len(detected)} player cell(s):')
for team, cell in detected:
    print(f'  Cell c{cell:02d}: {team}')


# Add player count stats to your results
results_df['player_cells_detected'] = results_df[
    [f'c{i:02d}' for i in range(1, 65)]
].apply(lambda row: (row > 0).sum(), axis=1)
